# Preprocessing the C-MAPSS FD001 dataset

**Notebook 01.** `notebooks/README.md` says how every notebook is laid out, who it is written for, how the notebooks keep what the paper says apart from what this project chose and what it measured, and where the code they call actually lives. Read it first.

## What this notebook is for

This notebook turns the raw NASA C-MAPSS text files into arrays that a graph neural network can be trained on, and explains every decision made along the way.

## What this notebook does *not* do

It does not train a model. Preprocessing is a separate stage on purpose: it has to be right before any model result means anything, and it is much easier to check on its own.

---

## The whole notebook in one list

Every step below takes the output of the step before it. Read this list first;
the rest of the notebook is these eight steps, expanded.

1. **Load** the three text files
2. **Diagnose** the sensors
3. **Select** 14 sensors
4. **Cap** the labels at 125
5. **Smooth** the readings
6. **Split** off 20 engines
7. **Rescale** every sensor
8. **Cut** into windows

Section 3 draws the same eight steps as a picture, with what goes into each and
what comes out, every count taken from a real run.

Step 8 produces an array of shape `(17731, 14, 30)` — 17 731 examples, 14 sensor
nodes each, the node's whole 30-cycle window — and the last part of step 8 then divides those
17 731 examples between the 80 training engines and the 20 held back.

How many of those 30 numbers a node **actually carries into the first graph
layer** is a question about the model, not about the data, so this notebook does
not answer it. Notebook 02 does, in the temporal stage: by default one number,
the last cycle, which is the `[batch, 14, 1]` the paper's Table 4 prints. Alongside it sits a
14 x 14 table saying which sensors are connected to which. Those two things are
what a graph neural network needs, and producing them correctly is the entire
job of this notebook.

**Why does the paper's graph layer get one number per sensor, when a node carries 30?** The paper's Table 4 gives the first graph layer one number per sensor, produced by a per-sensor recurrent network whose size the paper never states. By default this project's network passes on each sensor's most recent reading in its place (assumption A-07). Step 8 says why the pipeline still hands over the whole window, and notebook 02 builds the paper's own recurrent stage beside the stand-in.

## 1. Background

### The problem

A jet engine wears out gradually. **Remaining Useful Life (RUL)** is the number of flight cycles an engine will still run before it fails. Each row of NASA's files is "a snapshot of data taken during a single operational cycle" (NASA Prognostics Center of Excellence, 2008), and the paper describing the dataset takes "one measurement snapshot per flight" (Saxena et al., 2008, p. 5). This project reads one *flight cycle* as one flight; neither source says it in those words, as the documentation's Data page explains. The cycle is the only unit of time in this project. (Corrected 2026-10-19: this said a flight cycle is "one complete take-off, flight and landing", which neither source says.) Predicting RUL from sensor readings lets an airline plan maintenance before a failure happens.

### The dataset

**C-MAPSS** stands for *Commercial Modular Aero-Propulsion System Simulation*, a NASA program that simulates a large turbofan engine. NASA used it to build a public benchmark in which simulated engines run until failure while 21 sensors record the engine state once per cycle (Saxena et al., 2008).

The benchmark has four sub-datasets. This notebook uses **FD001**, the simplest: one operating condition and one fault mode, 100 training engines and 100 test engines. Starting with the simplest case is deliberate — a pipeline that is wrong here would be wrong everywhere, and the mistake is far easier to find.

| | FD001 | FD002 | FD003 | FD004 |
|---|---|---|---|---|
| Operating conditions | 1 | 6 | 1 | 6 |
| Fault modes | 1 | 1 | 2 | 2 |
| Training engines | 100 | 260 | 100 | 249 |
| Test engines | 100 | 259 | 100 | 248 |

### Training data and test data are shaped differently

This is the single most important thing to understand about C-MAPSS:

- **Training engines run to failure.** The last recorded row for an engine *is* the failure, so the remaining life at that row is zero by definition. No label file is needed.
- **Test engines stop early.** Recording ends at some point before failure. A separate file, `RUL_FD001.txt`, gives one number per test engine: how much life was left at its last recorded row.

Everything about how labels are computed follows from this difference.

### Why a *graph*

A turbofan is a chain of components. Air enters the fan, is compressed, is burned with fuel, and drives the turbines on the way out. A change in one part shows up in the sensors of the next. Treating the sensors as unrelated numbers throws that structure away.

A **graph** is a set of points, called **nodes**, with connections between them, called **edges**. Here each sensor is a node, and an edge means "these two measurements are physically related". A **graph neural network** uses those edges to decide which sensors may exchange information (Kipf & Welling, 2017; Corso et al., 2024).

## 2. Setup

The cell below imports everything the rest of the notebook uses, and adds `src/` to Python's search path so that the package imports from a clone that has not been installed with `pip install -e .`. `notebooks/README.md` explains why the computation lives in `src/` rather than here.

`help(apply_ewma)` in an empty cell prints one line saying what a function does. The full explanation lives in the `README.md` beside the code, here `src/thesis_gnn_rul_xai/data/smoothing/README.md` — every folder under `data/` has its own page.

In [ ]:
from __future__ import annotations

import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML

# Make the package importable when the notebook is opened directly from a
# clone that has not been installed with `pip install -e .`.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from thesis_gnn_rul_xai.config import PreprocessingConfig
from thesis_gnn_rul_xai.data.labeling import apply_piecewise_rul
from thesis_gnn_rul_xai.data.loading import load_subset
from thesis_gnn_rul_xai.data.normalization import (
    CONDITION_COL,
    OperatingConditionEncoder,
    SensorNormalizer,
)
from thesis_gnn_rul_xai.data.pipeline import run_preprocessing
from thesis_gnn_rul_xai.data.sensors import (
    PAPER_EXCLUDED_SYMBOLS,
    PAPER_SENSOR_COLUMNS,
    PAPER_SENSOR_SYMBOLS,
    SENSORS,
    column_to_symbol,
    detect_uninformative_sensors,
)
from thesis_gnn_rul_xai.data.smoothing import apply_ewma
from thesis_gnn_rul_xai.data.windowing import (
    make_last_windows,
    make_windows,
    split_units_by_id,
)
from thesis_gnn_rul_xai.docsite.figures import render_figure
from thesis_gnn_rul_xai.graph.adjacency import (
    adjacency_report,
    build_physics_adjacency,
    normalize_adjacency,
)
from thesis_gnn_rul_xai.utils.logging_setup import configure_logging
from thesis_gnn_rul_xai.utils.seeding import set_global_seed

configure_logging(logging.INFO)
set_global_seed(42)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (10, 4), "font.size": 9})

print(f"Project root: {PROJECT_ROOT}")
print(f"Python       : {sys.version.split()[0]}")
print(f"NumPy        : {np.__version__}")
print(f"pandas       : {pd.__version__}")

### The configuration object

Every setting that controls the pipeline is collected in one object rather than scattered through the code. Saving that object next to a result records exactly which settings produced it.

Settings marked **ASSUMPTION** below are *not* stated in the paper. They are choices made here, and each is explained in its own file in `docs/assumptions/`.

In [ ]:
config = PreprocessingConfig(
    subset="FD001",
    window_size=30,        # From the paper, Table 6.
    window_stride=1,       # A-05: not stated in the paper.
    rul_max=125,           # A-01: the paper never prints this value.
    clip_test_labels=True, # A-02: forced by the sigmoid output layer.
    normalization="zscore",  # A-03: the paper is silent. Worth 3.83 cycles
                             # against per-condition min-max (F-10).
    smoothing_alpha=0.5,   # A-04: the paper's sentence is self-contradictory.
    validation_fraction=0.2,     # A-06: the paper has no validation split.
    seed=42,
)

for field_name, value in config.to_dict().items():
    if field_name != "notes":
        print(f"  {field_name:22s} {value}")

## 3. Locating the data

This notebook runs on the real NASA measurements and nothing else. There is no stand-in dataset: a pipeline that appears to work on invented signals tells you nothing about whether it works on engines.

If the FD001 files are not in `data/raw/`, the next cell stops with a link to the download. `data/raw/README.md` says where to download them, and `python scripts/prepare_data.py --check` reports from the command line which of them are present.

In [ ]:
data_dir = PROJECT_ROOT / "data" / "raw"
required = [f"train_{config.subset}.txt", f"test_{config.subset}.txt", f"RUL_{config.subset}.txt"]
missing = [name for name in required if not (data_dir / name).is_file()]

if missing:
    raise FileNotFoundError(
        f"The {config.subset} files are not in {data_dir}: {missing}.\n\n"
        "This notebook runs on the real NASA measurements and nothing else. "
        "Download the 'Turbofan Engine Degradation Simulation Data Set' from\n\n"
        "    https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/\n\n"
        f"and put the three {config.subset} files into {data_dir}. If you keep the "
        "dataset elsewhere, set data_dir to that folder instead."
    )

print(f"Using the real NASA C-MAPSS files for {config.subset}.")
print(f"Data directory: {data_dir}")

### The whole pipeline in one picture

The list above names the eight steps. The figure below draws them, and adds the
thing a list cannot show: what the data actually *is* after each one — how many
rows, how many engines, what shape the array has.

None of those numbers is written down. The figure runs the pipeline as it is
drawn and prints what came back, so it cannot quietly fall out of step with the
code the way a hand-drawn diagram would. It is produced by
`draw_notebook_map()` in `src/thesis_gnn_rul_xai/docsite/figures.py`, the same
file that draws every diagram on the project's documentation website
(`python scripts/docs_ui.py`).

Three things to look for while reading it:

- **The two columns are the training engines and the test engines.** They
  arrive in different files, they are labelled by different rules, and step 8
  cuts them up differently. Almost everything in this notebook follows from
  that one difference.
- **The three steps in orange are the leakage-critical ones.** *Leakage* here
  means any path by which information about the held-back or test engines could
  reach a model that is supposed never to have seen them. Those three steps are
  where such a path would open, and their order is what closes it.
- **The graph, at the bottom, hangs off the side.** It is read from the paper's
  Table 3 rather than measured from the engines, so no arrow leads into it from
  the chain above.


In [ ]:
# Drawn from a real run of the pipeline, so the counts below are the same
# ones this notebook arrives at step by step in the sections that follow.
#
# It is handed over as HTML rather than as an SVG display object: the
# documentation site renders this notebook with nbconvert, which drops an
# image/svg+xml output and keeps a text/html one.
#
# That run would otherwise print a dozen progress lines above the picture.
# The steps below show the same log properly, one step at a time, so it is
# quietened here and here only.
package_log = logging.getLogger("thesis_gnn_rul_xai")
package_log.setLevel(logging.ERROR)
try:
    display(HTML(render_figure("notebook-map")))
finally:
    package_log.setLevel(logging.INFO)

## 4. The eight steps

Each of the eight subsections below is one step, and each takes the output of
the one before it. The order is not a matter of taste: **steps 5, 6 and 7 have
to happen in that order**, and the picture in section 3 marks them because
getting them the wrong way round produces good-looking numbers that mean
nothing.

Every step is written the same way: what it does, why it is necessary, and a
check confirming it did what was intended.

One thing worth saying before starting. Steps 6 and 7 look like bookkeeping and
are the two most dangerous steps here. Step 6 decides which engines are held
back, and step 7 measures the numbers used to rescale every sensor. If those
happen in the other order, the held-back engines help decide the scale they are
later judged against, and the score they produce is no longer an honest
estimate of anything.

### Step 1 — Load the raw files

#### What a raw file looks like

Each row describes one engine at one flight cycle, in 26 columns:

| Column | Meaning |
|---|---|
| 1 | Engine number (an identifier, not a measurement) |
| 2 | Cycle number, counted from when the engine was new |
| 3-5 | Three operational settings: altitude, Mach number, throttle angle |
| 6-26 | Twenty-one sensor readings |

There is no header row, and the columns are separated by spaces.

#### How labels are computed

For **training** engines, which run to failure:

$$\text{RUL}(t) = t_{\text{last}} - t$$

For **test** engines, which stop early, the label file supplies the missing piece:

$$\text{RUL}(t) = (t_{\text{last}} - t) + \text{RUL}_{\text{official}}$$

Worked example: an engine's last recorded cycle is 150 and the label file says 47 cycles remained at that moment. At cycle 130 the engine still had $(150 - 130) + 47 = 67$ cycles left.

In [ ]:
data = load_subset(config.subset, data_dir=data_dir)
print()
display(data.summary())

In [ ]:
# The first rows of the first engine. Note that `rul` counts down by one each
# cycle and that the operational settings barely move: FD001 has a single
# operating condition.
columns_to_show = ["unit", "cycle", "op_setting_1", "op_setting_2", "op_setting_3",
                   "sensor_2", "sensor_3", "sensor_4", "rul"]
display(data.train[columns_to_show].head(5))
print("...")
display(data.train[data.train["unit"] == 1][columns_to_show].tail(3))

**Check.** The final label of every training engine must be exactly zero: the last recorded cycle *is* the failure. And for every test engine, the label computed at its last recorded cycle must equal the number in the official label file.

In [ ]:
final_training_labels = data.train.groupby("unit")["rul"].last()
assert (final_training_labels == 0).all(), "A training engine does not end at RUL = 0."
print(f"OK  All {len(final_training_labels)} training engines end at RUL = 0.")

computed_last = data.test.groupby("unit")["rul"].last().sort_index()
official = data.test_rul.sort_index()
assert (computed_last.to_numpy() == official.to_numpy()).all(), "Test labels disagree."
print(f"OK  All {len(official)} test-engine labels match the official label file.")
print(f"    Official remaining life ranges from {official.min()} to {official.max()} cycles.")

In [ ]:
# How long do engines live? The spread is wide, which is what makes the task
# non-trivial: age alone is a poor predictor of remaining life.
lifetimes = data.train.groupby("unit")["cycle"].max()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].hist(lifetimes, bins=20, color="#4C72B0", edgecolor="white")
axes[0].set_xlabel("Total lifetime (flight cycles)")
axes[0].set_ylabel("Number of engines")
axes[0].set_title("Training engine lifetimes")
axes[0].axvline(lifetimes.median(), color="#C44E52", linestyle="--",
                label=f"median = {lifetimes.median():.0f}")
axes[0].legend()

test_lengths = data.test.groupby("unit")["cycle"].max()
axes[1].hist(test_lengths, bins=20, color="#55A868", edgecolor="white")
axes[1].set_xlabel("Recorded length (flight cycles)")
axes[1].set_ylabel("Number of engines")
axes[1].set_title("Test engine recorded lengths")
axes[1].axvline(config.window_size, color="#C44E52", linestyle="--",
                label=f"window = {config.window_size}")
axes[1].legend()

fig.tight_layout()
plt.show()

print(f"Shortest test history: {test_lengths.min()} cycles.")
print(f"Window size          : {config.window_size} cycles.")
if test_lengths.min() < config.window_size:
    print("NOTE: at least one test engine is shorter than the window and will be padded.")
else:
    print("Every test engine is long enough for one full window.")

### Step 2 — Which sensors carry information?

Some of the 21 sensors never change during an engine's entire life. A flat line
contains no information about degradation and cannot help any model.

Wen et al. (2026) keep 14 sensors and drop 7. The next cell checks that decision **against the data itself** rather than taking it on trust. Any disagreement is a finding worth reporting, not something to hide.

#### What the check finds on the real FD001 data

Two things, both worth a sentence in the thesis:

- **Exactly six sensors are perfectly constant**, and they are exactly the six the paper *names*. The seventh sensor missing from its adjacency matrix, `NRc`, is **not** constant — it takes over six thousand distinct values. It must have been dropped for another reason, most plausibly redundancy, since the corrected core speed is derived from the physical core speed `Nc`, which the paper keeps. The paper's sentence merges two different reasons for exclusion (**assumption A-10**).
- **`P15` is kept by the paper but takes only two distinct values** in the entire training set. It cannot represent gradual wear. It is kept here anyway, because the adjacency matrix is defined over the paper's fixed 14 sensors and removing a node would change the graph. The pipeline reports it rather than silently correcting it (**finding F-01**).

The right test here is the *number of distinct values*, not the size of the range: `Nf` has a smaller relative range than `P15` but spreads it over 53 levels and is genuinely informative.

#### A discrepancy in the paper

Section 3.2.2 of the paper says "seven sensors are removed" but then names only six: T2, P2, epr, farB, Nfdmd and PCNRdmd. Comparing the full sensor list in their Table 2 with the 14 sensors in their adjacency matrix in Table 3 shows the missing seventh is **NRc**, the corrected core speed. This project follows Table 3, because that is the list the model actually consumes (assumption **A-10**).

#### What the 21 sensors actually measure

Before deciding which sensors to keep, it is worth knowing what they are.

A turbofan is a chain, and the sensors are placed along it. Air enters the
**fan**, passes through the **low-pressure compressor (LPC)** and then the
**high-pressure compressor (HPC)**, is burned with fuel in the **combustor**,
and drives the **high-pressure turbine (HPT)** and then the **low-pressure
turbine (LPT)** on its way out. A large part of the air bypasses the core
entirely through the **bypass duct**, and on a modern engine that bypass flow
provides most of the thrust.

The names follow a convention, which makes them far easier to read once you
know it:

| Starts with | Measures | Example |
|---|---|---|
| `T` | total temperature | `T24`, temperature at the LPC outlet |
| `P` | pressure | `P30`, pressure at the HPC outlet |
| `N` | rotational speed of a shaft | `Nf`, the fan shaft's speed |
| `W` | mass flow rate | `W31`, coolant bled to the HPT |
| `epr`, `phi`, `BPR`, `farB` | ratios, so no units | `BPR`, the bypass ratio |

The number in a name is the **station** — a standard numbering for positions
along the gas path, increasing downstream. So `T24` is measured early, just
after the low-pressure compressor, and `T50` much later, after the low-pressure
turbine. Two names ending in `_dmd` are *demanded* values: what the control
system asked for rather than what was measured.

The units, spelled out:

- **degR** — degrees Rankine, an absolute temperature scale (0 degR is absolute
  zero, and one Rankine degree is the same size as one Fahrenheit degree).
- **psia** — pounds per square inch, absolute; a pressure measured against a
  vacuum rather than against the surrounding air.
- **rpm** — revolutions per minute.
- **pps/psi** — pounds of fuel per second, per psi of pressure.
- **lbm/s** — pounds of mass per second, a flow rate.

The table below is printed from `SENSORS` in `data/sensors/`, which is the list
the code itself uses, so it cannot drift away from what the pipeline does.


In [ ]:
# Every sensor, printed from the code's own catalogue rather than transcribed.
catalogue = pd.DataFrame(
    [
        {
            "symbol": sensor.symbol,
            "column": sensor.column,
            "measures": sensor.description,
            "unit": sensor.unit,
            "engine station": sensor.station,
            "kept by the paper": sensor.symbol in PAPER_SENSOR_SYMBOLS,
        }
        for sensor in SENSORS
    ]
)
display(catalogue)

print(f"{len(catalogue)} sensors in total, "
      f"{int(catalogue['kept by the paper'].sum())} kept by the paper, "
      f"{int((~catalogue['kept by the paper']).sum())} dropped.")


In [ ]:
diagnosis = detect_uninformative_sensors(data.train)

print("Detected from the data")
print("-" * 60)
print(f"  Constant (no variation at all) : {diagnosis['constant']}")
print(f"  Low resolution                 : {diagnosis['low_resolution']}")
print(f"  Informative                    : {len(diagnosis['informative'])} sensors")

print("\nStated by Wen et al. (2026)")
print("-" * 60)
print(f"  Dropped ({len(PAPER_EXCLUDED_SYMBOLS)}) : {list(PAPER_EXCLUDED_SYMBOLS)}")
print(f"  Kept    ({len(PAPER_SENSOR_SYMBOLS)}) : {list(PAPER_SENSOR_SYMBOLS)}")

detected_useless = set(diagnosis["constant"]) | set(diagnosis["low_resolution"])
paper_dropped = set(PAPER_EXCLUDED_SYMBOLS)

print("\nComparison")
print("-" * 60)
if detected_useless == paper_dropped:
    print("  The data agrees exactly with the paper's selection.")
else:
    only_data = sorted(detected_useless - paper_dropped)
    only_paper = sorted(paper_dropped - detected_useless)
    if only_data:
        print(f"  KEPT by the paper but uninformative here : {only_data}")
        print("    -> Reported, not silently removed. The adjacency matrix is")
        print("       defined over the paper's fixed 14 sensors, so changing the")
        print("       selection would break the graph.")
    if only_paper:
        print(f"  DROPPED by the paper but informative here : {only_paper}")
        print("    -> The paper had another reason, most likely redundancy with")
        print("       another sensor rather than lack of variation.")

In [ ]:
# A per-sensor variation table, ordered so the flat signals appear first.
rows = []
for sensor in SENSORS:
    values = data.train[sensor.column]
    rows.append({
        "symbol": sensor.symbol,
        "description": sensor.description[:38],
        "unit": sensor.unit,
        "min": values.min(),
        "max": values.max(),
        "range": values.max() - values.min(),
        "distinct": values.nunique(),
        "kept_by_paper": sensor.symbol in PAPER_SENSOR_SYMBOLS,
    })
variation = pd.DataFrame(rows).sort_values("range").reset_index(drop=True)
display(variation.style.format({"min": "{:.3f}", "max": "{:.3f}", "range": "{:.4f}"}))

#### Every sensor, drawn against time

The table above reports which sensors never move. The picture below shows the
same thing in a form you can check at a glance.

Each panel is one of the 21 sensors, drawn over the whole life of six engines:
one in red, so a single history can be followed from new to failed, and five in
grey behind it, so it is clear the behaviour belongs to the sensor rather than
to one unlucky engine.

What to look for:

- **The grey-backed panels** are the six sensors that never change. The line is
  flat because the number is the same at every cycle of every engine. A flat
  line cannot indicate wear, whatever model is put on top of it.
- **The amber-backed panel** is `P15`, which moves between exactly two values
  and nothing in between. The paper keeps it, so this project keeps it too, and
  reports the problem instead of quietly dropping it (finding F-01).
- **The rest drift**, most of them steadily up or down across an engine's life.
  That drift is the degradation the model has to learn to read.

One caution about the y-axis. Each panel is scaled to its own sensor, because
the sensors measure different quantities on wildly different scales -- a
temperature near 1400 next to a ratio near 8. So the **shape** of a line is
comparable between panels; its **height** is not. A panel that looks dramatic
may be showing a very small movement blown up to fill the space, which is
exactly why the number of distinct values is printed in each corner rather than
left to the eye.

In [ ]:
# Every sensor against time, so that a flat one is visible rather than asserted.
# Six engines per panel: one in red to follow, five in grey to show the
# behaviour is the sensor's and not one engine's.
show_units = [int(unit) for unit in np.unique(data.train["unit"])[:6]]
highlight_unit = show_units[0]
histories = {unit: data.train[data.train["unit"] == unit] for unit in show_units}

constant_symbols = set(diagnosis["constant"])
two_level_symbols = set(diagnosis["low_resolution"])

fig, axes = plt.subplots(3, 7, figsize=(15, 7.4))
for axis, sensor in zip(axes.ravel(), SENSORS):
    for unit in show_units:
        leading = unit == highlight_unit
        axis.plot(histories[unit]["cycle"], histories[unit][sensor.column],
                  color="#C44E52" if leading else "#AEB6C2",
                  linewidth=1.5 if leading else 0.6,
                  zorder=3 if leading else 1)

    distinct = int(data.train[sensor.column].nunique())
    if sensor.symbol in constant_symbols:
        axis.set_facecolor("#ECECEC")
        note = "never moves"
    elif sensor.symbol in two_level_symbols:
        axis.set_facecolor("#FAF0DC")
        note = f"{distinct} values only"
    else:
        note = f"{distinct:,} values"

    kept = sensor.symbol in PAPER_SENSOR_SYMBOLS
    axis.set_title(sensor.symbol if kept else f"{sensor.symbol}  (dropped)",
                   fontsize=9.5, fontweight="bold" if kept else "normal",
                   color="#1A1A1A" if kept else "#9A9A9A")
    axis.text(0.08, 0.06, note, transform=axis.transAxes, fontsize=7.5,
              color="#555555")
    axis.set_xticks([])
    axis.tick_params(axis="y", labelsize=6)
    # Without this, a sensor whose values sit close together is labelled
    # "+2.16e1" plus an offset, which hides the numbers themselves.
    axis.ticklabel_format(axis="y", useOffset=False, style="plain")

# The two colours, said outright rather than left to the title.
key = [
    plt.Line2D([0], [0], color="#C44E52", linewidth=1.8,
               label=f"engine {highlight_unit}: one engine followed from new to failed"),
    plt.Line2D([0], [0], color="#AEB6C2", linewidth=1.0,
               label=f"{len(show_units) - 1} more engines: drawn behind it, so a shape "
                     "that repeats is the sensor's and not one engine's"),
]
fig.suptitle("All 21 sensors, drawn over the whole life of six engines",
             fontsize=12.5)
fig.legend(handles=key, loc="upper center", ncol=2, frameon=False,
           fontsize=9.5, bbox_to_anchor=(0.5, 0.955))
fig.supxlabel("Flight cycle: new on the left, failed on the right. "
              "Each panel spans the whole life of every engine in it.",
              fontsize=9.5)
fig.tight_layout(rect=(0, 0.03, 1, 0.925))
plt.show()

print(f"Never move at all ({len(constant_symbols)}): "
      f"{sorted(constant_symbols)}")
print(f"Move between two values only ({len(two_level_symbols)}): "
      f"{sorted(two_level_symbols)}")
print(f"Carry real variation ({len(diagnosis['informative'])}): "
      f"{len(diagnosis['informative'])} sensors")
print()
print("The grey panels are the flat ones. They record an identical number at")
print("every cycle of every engine, so no model can learn anything from them,")
print("and the paper drops all six. The amber panel is P15, kept by the paper")
print("in spite of taking two values; it stays, and is reported (F-01).")

### Step 3 — Keep the paper's 14 sensors, in the paper's order

The order is not cosmetic. The adjacency matrix in the paper's Table 3 is indexed **by position**: row 1 is T24, row 2 is T30, and so on. If the sensor columns were arranged differently, the graph would connect the wrong sensors and nothing would report an error. The order is therefore fixed in the source code and checked by an assertion whenever the package is imported.

In [ ]:
sensor_columns = list(PAPER_SENSOR_COLUMNS)

print(f"{'node':>4}  {'symbol':<9} {'column':<11} description")
print("-" * 72)
for index, (symbol, column) in enumerate(zip(PAPER_SENSOR_SYMBOLS, sensor_columns)):
    description = next(s.description for s in SENSORS if s.symbol == symbol)
    print(f"{index:>4}  {symbol:<9} {column:<11} {description}")

assert [column_to_symbol(c) for c in sensor_columns] == list(PAPER_SENSOR_SYMBOLS)
print("\nOK  Column order matches the row order of the paper's Table 3.")

### Step 4 — Build the label: the piecewise-linear target

#### The problem with a plain countdown

If the label were simply "cycles until failure", a brand-new engine in a long history would carry a label of 300 or more. But a new engine gives the model no way to tell 300 from 280: almost no wear has happened yet, so the sensors look the same either way. Asking a model to distinguish them forces it to guess, and that guessing adds noise to every prediction it makes.

#### The standard solution

Hold the label at a ceiling while the engine is healthy, and let it fall only once real degradation begins (Heimes, 2008):

$$\text{RUL}(t) = \min\big(t_{\text{last}} - t,\; \text{RUL}_{\max}\big)$$

Drawn against time this is a flat line followed by a straight decline. The point where they meet is often called the **change point**.

#### The ceiling used here

**The ceiling of 125 is this project's choice.** The paper gives this formula as its Equation 19 but never prints the value of $\text{RUL}_{\max}$ (assumption A-01). Two things point to 125: Section 4.2 of the paper describes predicted life holding "around 125" while an engine is healthy, and Jin et al. (2025) use 125 on the same dataset. Heimes originally reported that anything between about 120 and 130 works well.

**The same ceiling is put on the test labels** (assumption A-02). This is not optional here: the output layer in the paper's Table 4 is a sigmoid function, so the model *cannot* produce a value above the ceiling. Leaving test labels uncapped would charge the model for an error its own architecture makes impossible to avoid.

In [ ]:
train_labeled = apply_piecewise_rul(data.train, rul_max=config.rul_max, output_column="rul_capped")
test_labeled = apply_piecewise_rul(data.test, rul_max=config.rul_max, output_column="rul_capped")

example_unit = int(train_labeled["unit"].iloc[0])
history = train_labeled[train_labeled["unit"] == example_unit]

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(history["cycle"], history["rul"], label="plain countdown", color="#8C8C8C")
ax.plot(history["cycle"], history["rul_capped"], label=f"capped at {config.rul_max}",
        color="#C44E52", linewidth=2)
ax.axhline(config.rul_max, color="#4C72B0", linestyle=":", label="the ceiling")
ax.set_xlabel("Flight cycle")
ax.set_ylabel("Remaining Useful Life (cycles)")
ax.set_title(f"Engine {example_unit}: the label the model is asked to predict")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

corner = int((history["rul_capped"] < config.rul_max).idxmax())
print(f"Engine {example_unit} ran {int(history['cycle'].max())} cycles.")
print(f"Its label sits flat at {config.rul_max} until cycle "
      f"{int(history.loc[corner, 'cycle'])}, then counts down to zero.")

#### What a histogram of labels is counting

The two pictures after this one count labels. They are easy to misread, so this
one builds a single bar in front of you, starting from one engine.

A **row** is one engine at one flight cycle. Engine 1 ran 192 cycles, so it puts
192 rows into the table, and each row carries its own label: 191 at the first
cycle, then 190, and so on down to 0 at the last.

A histogram throws away **when** a row happened and keeps only **what label it
carried**. One bar answers one question: how many rows carried a label in this
range? Nothing about time survives the counting.

Below, that is done twice. The tallies are turned on their side so that each bar
sits level with the stretch of labels it counts.

- **One engine, on the top row.** Its line falls steadily from 191 to 0, passing
  every label on the way exactly once. So with bars ten labels wide, every bar is
  exactly ten rows long. Read that again if it seems too simple: the bar is not
  measuring the engine's health, or how long it lived at that label. It is
  counting cycles, and one engine spends one cycle at each label.
- **Eight engines, on the bottom row.** The same bars, eight times longer,
  because eight lines now cross each band instead of one. They stay that length
  until the shorter-lived engines stop reaching the higher labels -- and there
  the bars start to shorten.

That is the whole mechanism, and it is what gives the next picture its shape: a
long flat plateau across the labels every engine reaches, and a decline across
the labels only the long-lived engines reach.

In [ ]:
# Where the bars in the next two pictures come from, built up from one engine.
# Left column: engines counting down to failure. Right column: the same rows
# tallied, turned on its side so each bar sits level with the labels it counts.
band_low, band_high = 100, 110          # one bar's worth of labels
bar_width = band_high - band_low
edges = np.arange(0, 370, bar_width)

one_unit = int(np.unique(train_labeled["unit"])[0])
eight_units = [int(unit) for unit in np.unique(train_labeled["unit"])[:8]]


def countdown_and_tally(line_axis, tally_axis, units, heading):
    """Draw one group of engines as falling lines, and as a tally beside them."""
    rows = train_labeled[train_labeled["unit"].isin(units)]
    for unit in units:
        history = rows[rows["unit"] == unit]
        line_axis.plot(history["cycle"], history["rul"], linewidth=1.1,
                       color="#8C8C8C")
    line_axis.axhspan(band_low, band_high, color="#C44E52", alpha=0.18)
    line_axis.set_ylabel("Remaining Useful Life (cycles)")
    line_axis.set_xlim(0, 300)
    line_axis.set_ylim(0, 300)
    line_axis.set_title(heading, fontsize=10.5)

    _, _, bars = tally_axis.hist(rows["rul"], bins=edges,
                                 orientation="horizontal", color="#B4B4B4")
    for bar, low in zip(bars, edges[:-1]):
        if low == band_low:
            bar.set_facecolor("#C44E52")
    # Both tallies share one scale, so "eight times longer" is visible.
    tally_axis.set_xlim(0, 8 * bar_width + 10)
    tally_axis.set_title("the same rows, tallied", fontsize=10.5)
    return int(((rows["rul"] >= band_low) & (rows["rul"] < band_high)).sum())


fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.4), sharey=True,
                         gridspec_kw={"width_ratios": [2.1, 1]})

one_row = countdown_and_tally(
    axes[0][0], axes[0][1], [one_unit],
    f"ONE engine (number {one_unit}), counting down to failure")
eight_rows = countdown_and_tally(
    axes[1][0], axes[1][1], eight_units,
    f"{len(eight_units)} engines, exactly the same picture")

label_box = {"facecolor": "white", "edgecolor": "none", "alpha": 0.85,
             "pad": 1.5}
axes[0][1].annotate(f"{one_row} rows: it passes each of\nthese "
                    f"{bar_width} labels exactly once",
                    xy=(one_row + 5, band_low), fontsize=9,
                    color="#C44E52", va="center", bbox=label_box)
axes[1][1].annotate(f"{eight_rows} rows\n= {len(eight_units)} engines "
                    f"x {bar_width} labels",
                    xy=(4, band_low + 26), fontsize=9, color="#C44E52",
                    bbox=label_box)
axes[1][0].set_xlabel("Flight cycle")
axes[1][1].set_xlabel("number of rows")

fig.suptitle("What one bar of a label histogram is counting", fontsize=12.5)
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

everyone = train_labeled[(train_labeled["rul"] >= band_low)
                         & (train_labeled["rul"] < band_high)]
print(f"Labels {band_low} to {band_high - 1}, which is {bar_width} labels wide:")
print(f"  engine {one_unit} alone            {one_row:>6} rows")
print(f"  the first {len(eight_units)} engines        {eight_rows:>6} rows"
      f"   = {len(eight_units)} x {bar_width}")
print(f"  all {train_labeled['unit'].nunique()} training engines  "
      f"{len(everyone):>6} rows"
      f"   = {train_labeled['unit'].nunique()} x {bar_width}")
print()
print("A bar's length is a head-count of engine-cycles whose label fell in its")
print("range. Nothing else is in it: not how long the engine lived, not how")
print("worn it was, not how many engines failed. Just how many cycles carried")
print("that label.")

In [ ]:
# The same 20 631 rows counted twice, before capping and after. Both panels are
# drawn on ONE vertical scale, which is the only way to see that the halves
# below the ceiling really are identical. The cost is that the ceiling bar no
# longer fits, so it is cut off and its height written beside it.
bins = np.arange(0, 370, 5)
above = int((train_labeled["rul"] > config.rul_max).sum())
at_ceiling = int((train_labeled["rul_capped"] == config.rul_max).sum())
plateau = int(((train_labeled["rul"] >= 0) & (train_labeled["rul"] < 5)).sum())
head_room = 1.12 * plateau

fig, axes = plt.subplots(2, 1, figsize=(9.5, 6.4), sharex=True, sharey=True)

axes[0].hist(train_labeled["rul"], bins=bins, color="#8C8C8C")
axes[0].axvspan(config.rul_max, 365, color="#C44E52", alpha=0.10)
axes[0].set_title("BEFORE capping: the plain countdown, one bar per 5 cycles")
label_box = {"facecolor": "white", "edgecolor": "none", "alpha": 0.85,
             "pad": 2}
axes[0].annotate(f"{above:,} rows sit above the ceiling,\n"
                 "thinning out to 361. Every one of\nthem moves.",
                 xy=(152, 0.60 * head_room), fontsize=9, color="#C44E52",
                 bbox=label_box)

axes[1].hist(train_labeled["rul_capped"], bins=bins, color="#C44E52")
axes[1].set_title(f"AFTER capping: all of them moved to exactly {config.rul_max}")
axes[1].annotate(f"{at_ceiling:,} rows in this one bar,\n"
                 f"{at_ceiling / plateau:.0f} times the height of the axis",
                 xy=(config.rul_max, head_room), xytext=(150, 0.62 * head_room),
                 arrowprops={"arrowstyle": "->", "color": "#C44E52"},
                 fontsize=9, color="#C44E52", bbox=label_box)
axes[1].set_xlabel("Remaining Useful Life (cycles)")

for axis in axes:
    axis.axvline(config.rul_max, color="#C44E52", linestyle=":", linewidth=1.2)
    axis.set_ylabel("number of rows")
    axis.set_ylim(0, head_room)

fig.suptitle("Both panels on one vertical scale, so the halves below the "
             "ceiling can be compared", fontsize=10.5, y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

print(f"Rows at or below the ceiling : {len(train_labeled) - above:,}  "
      "(unchanged by capping)")
print(f"Rows above the ceiling       : {above:,}  "
      f"({above / len(train_labeled):.1%}), all moved to {config.rul_max}")
print(f"Rows sitting at exactly {config.rul_max}  : {at_ceiling:,}"
      f"  = {above:,} moved + {at_ceiling - above} that were already there")
print()
print(f"The tallest bar that fits on the axis is {plateau}. The ceiling bar is")
print(f"{at_ceiling:,}, so it is drawn cut off rather than shrinking every other")
print("bar to a smudge. That single bar is the whole point of capping.")

#### How to read those two histograms

The two panels are **the same 20 631 training rows**, counted twice: once
before capping and once after. A bar's height is how many rows carry a label in
that five-cycle band.

**Both panels use one vertical scale**, which is what makes them comparable at
all. The price is that the bar at the ceiling does not fit: it holds 8 131 rows
against an axis that stops at about 560, so it is drawn cut off with its real
height written beside it. Rescaling to fit it instead would squash every other
bar to a smudge a pixel high, which is what the first version of this picture
did.

**The left half of both panels is identical, and that is the point.** Capping
only touches rows whose label is *above* 125, so every row below the ceiling
comes through untouched. If the two panels look the same up to 125, nothing has
gone wrong — that is exactly what capping is supposed to do.

**Why the flat plateau sits at 500.** Every engine counts down through every low
label exactly once: engine 1 has exactly one row labelled 7, one labelled 8, and
so on. With 100 engines and bars five cycles wide, that is `100 x 5 = 500` rows
per bar, for as long as every engine is still alive at that label.

**What changes is the right-hand tail.** Before capping, 8 031 rows carry labels
from 126 up to 361, thinning out as they go — those are the early, healthy
cycles of the longer-lived engines. After capping, every one of them is set to
exactly 125, so the whole tail collapses into a single bar.

**Why that bar is 8 131 and not 8 031.** The 8 031 rows that moved, plus the 100
that were *already* sitting at exactly 125 — one per engine, since each engine
passes through 125 on its way down.

**What this buys.** The tail was the part of the data the sensors cannot
distinguish: a row labelled 300 and a row labelled 250 look alike, because
neither engine has worn appreciably. Collapsing them removes a question the
model could only answer by guessing, and leaves the distinctions the
measurements actually support.

In [ ]:
# Apply the cap in place from here on.
train_frame = apply_piecewise_rul(data.train, rul_max=config.rul_max)
test_frame = (
    apply_piecewise_rul(data.test, rul_max=config.rul_max)
    if config.clip_test_labels else data.test.copy()
)

assert train_frame["rul"].max() <= config.rul_max
assert train_frame["rul"].min() == 0
print(f"OK  Training labels now lie in [{train_frame['rul'].min()}, {train_frame['rul'].max()}].")
print(f"OK  Test labels now lie in [{test_frame['rul'].min()}, {test_frame['rul'].max()}].")

### Step 5 — Smooth the signals

Sensor readings contain **measurement noise**: small, fast fluctuations unrelated to engine wear. Wear itself is slow and gradual. Removing the fast part makes the slow trend easier to see.

#### The filter

An **exponentially weighted moving average** replaces each reading with a blend of that reading and the smoothed value from the previous cycle:

$$S_t = \alpha X_t + (1 - \alpha) S_{t-1}$$

- $\alpha$ close to 1: almost no smoothing, the output follows the raw signal.
- $\alpha$ close to 0: strong smoothing, but the output reacts late to a real change.

The name comes from expanding the formula: every past reading contributes, with a weight that shrinks exponentially the further back it lies.

#### Two properties that keep the experiment honest

**It only looks backwards.** $S_t$ depends on $X_t$ and earlier values, never on later ones. A filter that averaged over future readings would leak information about the future into the present, and the scores would stop measuring what the model can do with the readings available at the time. Usually the model would appear to predict failure better than it really can, but nothing guarantees the direction: a test engine's record stops before failure, so its last window has no later readings to average over, and the filter would treat it differently from the training windows. This is a form of **data leakage** (Kapoor & Narayanan, 2023) and is checked by an automated test. (Corrected 2026-10-02: this said the model would appear to predict failure better than it really can; found by the search for that kind of claim after the review of the playground's information cards on 2026-09-26.)

**It is applied per engine.** No reading from one engine can influence another.

#### The value of $\alpha$

**Reading the smoothing strength as 0.5 is this project's choice** (assumption A-04). Section 3.2.3 of the paper says: *"S(t-1) indicates the previous smoothed value, which is equal to 0.5 in this study; α constitutes the smoothing parameter"*. As written, 0.5 is attached to the wrong symbol. Since $S_{t-1}$ is a different quantity at every time step, it cannot be a fixed number; the only reading that makes the sentence meaningful is $\alpha = 0.5$.

In [ ]:
train_smoothed = apply_ewma(train_frame, sensor_columns, alpha=config.smoothing_alpha)
test_smoothed = apply_ewma(test_frame, sensor_columns, alpha=config.smoothing_alpha)

# Show the effect on the three sensors that degrade most clearly.
show_symbols = ["T50", "Ps30", "W31"]
fig, axes = plt.subplots(len(show_symbols), 1, figsize=(10, 6), sharex=True)

for axis, symbol in zip(axes, show_symbols):
    column = sensor_columns[list(PAPER_SENSOR_SYMBOLS).index(symbol)]
    raw = train_frame[train_frame["unit"] == example_unit]
    smooth = train_smoothed[train_smoothed["unit"] == example_unit]
    axis.plot(raw["cycle"], raw[column], color="#B0B0B0", linewidth=0.8, label="raw")
    axis.plot(smooth["cycle"], smooth[column], color="#C44E52", linewidth=1.6,
              label=f"smoothed (alpha={config.smoothing_alpha})")
    axis.set_ylabel(symbol)
    axis.legend(fontsize=8, loc="upper left")

axes[-1].set_xlabel("Flight cycle")
axes[0].set_title(f"Engine {example_unit}: effect of the smoothing filter")
fig.tight_layout()
plt.show()

In [ ]:
# Quantify the effect: how much of the cycle-to-cycle jitter was removed?
def mean_absolute_step(frame: pd.DataFrame, column: str) -> float:
    """Average size of the change from one cycle to the next, over all engines."""
    return frame.groupby("unit")[column].diff().abs().mean()

rows = []
for symbol, column in zip(PAPER_SENSOR_SYMBOLS, sensor_columns):
    before = mean_absolute_step(train_frame, column)
    after = mean_absolute_step(train_smoothed, column)
    rows.append({
        "sensor": symbol,
        "jitter_before": before,
        "jitter_after": after,
        "reduction": 1 - after / before if before > 0 else 0.0,
    })
jitter = pd.DataFrame(rows)
display(jitter.style.format({"jitter_before": "{:.4f}", "jitter_after": "{:.4f}",
                             "reduction": "{:.1%}"}))
print(f"\nMedian reduction in cycle-to-cycle jitter: {jitter['reduction'].median():.1%}")

### Step 6 — Hold out the validation engines

Two steps happen in this section, and **the order between them matters**.

The 20 validation engines are chosen *first*, before any average, minimum
or maximum is computed. If the scaling were worked out over all 100
engines and 20 of them were held back afterwards, one of those held-back
engines could be the one defining a sensor's largest value, and the
held-back set would have quietly helped to prepare the data it is supposed
to judge. *Applying the split from step 6*, at the end of this section, explains
what the split is for; here it is only decided.

In [ ]:
# The validation engines are chosen HERE, before any statistic is learned.
# Fitting the encoder or the scaler on all 100 engines and only afterwards
# holding 20 of them out would let a held-out engine define a sensor's range,
# which is the same kind of leak that keeping the test set separate prevents.
train_units, validation_units = split_units_by_id(
    np.unique(train_smoothed["unit"].to_numpy()),
    validation_fraction=config.validation_fraction,
    seed=config.seed,
)
fitting_frame = train_smoothed[train_smoothed["unit"].isin(train_units)]
print(f"Statistics will be fitted on {len(train_units)} training engines; "
      f"{len(validation_units)} engines are held out.")

condition_encoder = OperatingConditionEncoder(seed=config.seed).fit(fitting_frame)
train_conditioned = condition_encoder.transform(train_smoothed)
test_conditioned = condition_encoder.transform(test_smoothed)

print(f"\nOperating conditions found in training data: {condition_encoder.n_detected}")
print("Rows per condition (training):")
print(train_conditioned[CONDITION_COL].value_counts().sort_index().to_string())
print("\nRows per condition (test):")
print(test_conditioned[CONDITION_COL].value_counts().sort_index().to_string())

### Step 7 — Rescale every sensor

#### Why rescaling is needed

The 14 sensors measure different physical quantities on wildly different scales: a turbine outlet temperature near 1400 degrees Rankine, a bypass ratio near 8.4, a coolant bleed flow near 39 pounds per second. Fed to a neural network unchanged, the large numbers dominate simply because they are large.

#### What an operating condition is

The three "operational setting" columns describe how the aircraft was flown: altitude, Mach number, and throttle position. Flying higher or faster changes every sensor immediately and by a lot — and that change has nothing to do with wear.

In FD002 and FD004 the engines fly under six conditions, and the jumps between them are far larger than the slow drift caused by degradation. Scaling all six together buries the wear signal. Scaling each condition separately removes the flight-related jumps and leaves the wear visible.

**FD001 has only one operating condition**, so per-condition and global scaling give identical results here. The general version is implemented and tested now so that the code is already correct when FD002 and FD004 are added.

Conditions are found with **k-means clustering**, which sorts points into a chosen number of groups by repeating two steps until the centres stop moving: put every point in the group whose centre is nearest, then move each centre to the average of the points now in its group. The project runs this loop through scikit-learn's `KMeans` at its default `algorithm` setting when the training engines show more than one operating condition; on FD001, which has one, it is skipped and no clustering runs at all. The name *k-means* comes from MacQueen (1967, p. 281), but this loop is not the procedure he gave the name to: he describes it as one suggested to him by Forgy and Jennrich (p. 294). His own k-means takes the points one at a time, adds each to the group with the nearest average, and updates only that group's average (p. 283). These are the proceedings' page numbers; in the copy in `docs/references/` they are pages 1, 14 and 3. Until 2026-10-02 this paragraph credited the loop to MacQueen; the error was found when the cited papers were collected into `docs/references/` and checked against the places that cite them.

#### The rule that protects the whole experiment

> **All statistics come from the training engines only.**

Every minimum, maximum, mean, standard deviation and cluster centre is computed on training data and then applied *unchanged* to the test data.

If the maximum were computed over training and test data together, information about the test engines would enter the training process. The model would then be evaluated on data it had partly seen, and the reported error would no longer measure how it does on genuinely new engines. Usually it would come out smaller, but nothing guarantees the direction: a network trained on differently rescaled readings is a different network, and it could do better or worse (assumption A-25). This is **data leakage**, and it is among the most common causes of published results that cannot be reproduced (Kapoor & Narayanan, 2023). (Corrected 2026-10-02: this said the reported error would be smaller than the error on genuinely new engines; found by the search for that claim after the review of the playground's information cards on 2026-09-26.)

**The rescaling is this project's choice, not the paper's** (assumption A-03). The paper never describes its input scaling at all — the words "normalisation", "min-max" and "standardisation" do not appear in connection with the inputs, and operating conditions are never mentioned in its preprocessing section.

This notebook uses the **z-score**: each sensor's training mean is subtracted and the result divided by its training spread (standard deviation), so a rescaled reading says how many spreads it sits from that sensor's average. Until 2026-08-29 the project used instead the per-condition min-max scaling of Jin et al. (2025), who describe their steps on the same dataset. Measured against it over five seeds, the z-score landed 3.83 cycles closer to the paper's reported error, and the project's rule for settings the paper leaves open — take the option nearest the paper's own figure — selected it (assumption A-22). That rule makes the choice depend on the figure it is later compared with, which is a weakness to state, not a strength. Either way this is a documented *substitution*, not a reconstruction of what the original authors did.

In [ ]:
normalizer = SensorNormalizer(method=config.normalization, columns=sensor_columns)
normalizer.fit(train_conditioned[train_conditioned["unit"].isin(train_units)])  # <-- training engines only

train_scaled = normalizer.transform(train_conditioned)
test_scaled = normalizer.transform(test_conditioned)

print("\nLearned scaling statistics (first five sensors):")
display(normalizer.statistics_.head(5).style.format(
    {"min": "{:.3f}", "max": "{:.3f}", "mean": "{:.3f}", "std": "{:.3f}"}))

In [ ]:
# The mean and the spread were learned from the 80 training engines
# only, so those are the only rows the scale is centred on. Reporting
# the three groups separately makes that visible.
def report_range(frame, label):
    values = frame[sensor_columns].to_numpy()
    print(f"{label:<34} [{values.min():+.4f}, {values.max():+.4f}]")

fitting_rows = train_scaled["unit"].isin(train_units)
report_range(train_scaled[fitting_rows], "Training engines (scale fitted here)")
report_range(train_scaled[~fitting_rows], "Validation engines (held out)")
report_range(test_scaled, "Test engines")

print()
print("The readings are no longer in the sensors' own units. Under A-03 each")
print("one is replaced by how many spreads it sits above or below its own")
print("sensor's mean, so 0 is an average reading and -11 is eleven spreads")
print("below average - which is what the two-level sensor P15 produces (F-01).")
print()
print("Nothing is clipped. An engine that never helped fit the statistics is")
print("allowed to fall outside anything seen while fitting them, and that is")
print("correct: it is a real difference, not an error to be hidden.")
print()
print("Before the fix recorded in docs/assumptions/A-25.md the mean and the spread")
print("were computed over all 100 engines, so the held-back engines helped set")
print("the scale by which they were later judged. That was the leak.")


In [ ]:
# Before and after, on a common axis, for the same three sensors.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
history_raw = train_smoothed[train_smoothed["unit"] == example_unit]
history_scaled = train_scaled[train_scaled["unit"] == example_unit]

for symbol in show_symbols:
    column = sensor_columns[list(PAPER_SENSOR_SYMBOLS).index(symbol)]
    axes[0].plot(history_raw["cycle"], history_raw[column], label=symbol)
    axes[1].plot(history_scaled["cycle"], history_scaled[column], label=symbol)

axes[0].set_title("Before scaling (physical units)")
axes[0].set_xlabel("Flight cycle"); axes[0].set_ylabel("Sensor value")
axes[0].set_yscale("log"); axes[0].legend(fontsize=8)

axes[1].set_title("After scaling (dimensionless)")
axes[1].set_xlabel("Flight cycle"); axes[1].set_ylabel("Scaled value")
axes[1].legend(fontsize=8)

fig.tight_layout()
plt.show()
print("The left panel needs a logarithmic axis to show three sensors at once.")
print("After scaling they are directly comparable, and the degradation trend is")
print("visible in all three.")

#### Demonstrating that no leakage happens

The claim above is worth testing rather than trusting. The next cell deliberately corrupts the test set by multiplying every sensor by one million, applies the fitted scaler to it, and then checks that the *training* data still scales to exactly the same numbers as before.

If the scaler were being refitted on whatever data it was shown, this check would fail.

In [ ]:
before = normalizer.transform(train_conditioned)[sensor_columns].to_numpy()

corrupted = test_conditioned.copy()
corrupted[sensor_columns] = corrupted[sensor_columns] * 1_000_000
_ = normalizer.transform(corrupted)

after = normalizer.transform(train_conditioned)[sensor_columns].to_numpy()

assert np.array_equal(before, after), "LEAKAGE: the scaler changed after seeing test data."
print("OK  Corrupting the test set by a factor of one million changed nothing")
print("    about how the training data is scaled. No leakage through this step.")

### Step 8 — Cut the histories into windows

A network needs a fixed-size input, but engine histories have different lengths. The histories are therefore cut into short, fixed-length pieces called **time windows**. One window is one example.

With a window of 30 cycles and a **stride** of 1, the first example covers cycles 1-30, the second covers 2-31, and so on.

Each window is labelled with the Remaining Useful Life at its **last** cycle. That matches how the model would be used: an operator has the most recent 30 cycles of readings and wants to know the remaining life *now*.

The paper gives a window of 30 in its Table 6 but no stride; 1 is used here (**assumption A-05**).

#### The shape of the result

The array is arranged for a graph neural network, where each sensor is a node:

$$(\text{examples},\; \text{sensors},\; \text{cycles per sensor}) = (N,\; 14,\; 30)$$

**Every node carries its whole 30-cycle window.** The paper's Table 4 gives the input to the first graph layer as `[batch, 14, 1]` — **one** number per sensor, produced by a per-sensor recurrent network whose size and depth the paper never states. Turning a sensor's 30 readings into that one number is the model's job, not the data's, so it happens in notebook 02, in a named stage in front of the graph layers. By default that stage keeps the most recent reading and drops the other 29 (assumption A-07); the paper's own recurrent stage is built beside it, with a size this project had to choose (assumption A-26).

**Why the pipeline no longer makes that decision.** Until 2026-09-09 it did: the pipeline cut each window down before handing it over, and the network read its width off the data. On 2026-08-28 deleting one line of preprocessing silently changed the network's first weight matrix from `(1, 32)` to `(30, 32)`, and two of the project's configuration comparisons reversed direction before anyone noticed the architecture had changed. With the decision made inside the model, where it is named, it cannot change without the model saying so.

**What the window decides here** is *which moments are eligible*. An engine's first example ends at cycle 30, not cycle 1, because a 30-cycle window has to fit behind it — which is exactly why 2 900 of the 20 631 rows produce no example.

#### Test data is treated differently, for a reason

Test histories stop before failure, and the official label describes exactly one moment: the last recorded cycle. The evaluation therefore uses **one window per test engine**, its final one. On the real FD001 this gives 100 predictions, which matches the "number of testing samples" column in the paper's Table 5.

In [ ]:
train_windows_all = make_windows(
    train_scaled, sensor_columns,
    window_size=config.window_size,
    stride=config.window_stride,
)
test_windows = make_last_windows(
    test_scaled, sensor_columns,
    window_size=config.window_size,
)

print()
print(f"Training windows : {train_windows_all}")
print(f"Test windows     : {test_windows}")
print()
print(f"Feature array shape: {train_windows_all.features.shape}")
print(f"  axis 0 = {train_windows_all.features.shape[0]:>6} examples")
print(f"  axis 1 = {train_windows_all.features.shape[1]:>6} sensors (graph nodes)")
print(f"  axis 2 = {train_windows_all.features.shape[2]:>6} number(s) per sensor (A-07)")
print(f"Memory: {train_windows_all.features.nbytes / 1024**2:.1f} MB as float32")

In [ ]:
# Check: one test window per engine, and its label is the official one.
assert len(test_windows) == test_scaled["unit"].nunique()
print(f"OK  Exactly one window per test engine ({len(test_windows)} engines).")

official_capped = np.minimum(data.test_rul.sort_index().to_numpy(), config.rul_max)
order = np.argsort(test_windows.units)
assert np.allclose(test_windows.targets[order], official_capped)
print("OK  Every test window carries the official (capped) label.")

In [ ]:
# What one example actually looks like: a window per sensor, and the one
# number the default temporal stage takes from it.
example_index = len(train_windows_all) // 2
example = train_windows_all.features[example_index]      # shape (14, 30)
last_cycle = example[:, -1]                              # what LastCycle keeps

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8),
                         gridspec_kw={"width_ratios": [1.5, 1]})

positions = list(range(len(PAPER_SENSOR_SYMBOLS)))
axes[0].barh(positions, last_cycle)
axes[0].set_yticks(positions)
axes[0].set_yticklabels(PAPER_SENSOR_SYMBOLS, fontsize=8)
axes[0].invert_yaxis()
axes[0].axvline(0.0, color="black", linewidth=0.8)
axes[0].set_xlabel("Scaled reading at the last cycle of the window")

engine = train_windows_all.units[example_index]
end_cycle = train_windows_all.end_cycles[example_index]
label = train_windows_all.targets[example_index]
title = ("One training example" + chr(10) +
         f"engine {engine}, ends at cycle {end_cycle}, label = {label:.0f}")
axes[0].set_title(title)

image = axes[1].imshow(example, aspect="auto", cmap="viridis")
axes[1].set_yticks(range(14))
axes[1].set_yticklabels(PAPER_SENSOR_SYMBOLS, fontsize=7)
axes[1].set_xticks([0, example.shape[1] - 1])
axes[1].set_xticklabels(["first cycle", "last cycle"])
axes[1].set_title(f"The same example as a matrix, shape {example.shape}")
fig.colorbar(image, ax=axes[1], label="scaled value")

fig.tight_layout()
plt.show()

print("Left: the one number per node that the default temporal stage "
      "takes,")
print("which is the last cycle of the window and the shape Table 4 prints.")
print("Right: the whole window the pipeline actually hands over, all 30")
print("cycles of it. Which of them a node carries is decided in the model")
print("(A-07), and the paper's own stage would read all thirty (A-26).")

### Applying the split from step 6

Step 6 chose *which* engines to hold back. Now that the
histories have been cut into windows, that decision is applied: every
window inherits the side its engine was assigned to.

A **validation set** is a portion of the training data that is held back and never used to update the model. It is used to decide when to stop training and to compare settings, so that those decisions are not made by looking at the test set.

#### The split must be by engine, not by window

This is easy to get wrong and expensive when you do. With a window of 30 and a stride of 1, two neighbouring windows from the same engine share **29 of their 30 cycles**, and their labels differ by one cycle, or not at all while both sit at the ceiling of 125. If windows were split at random, nearly identical examples would land on both sides: the model would effectively be tested on data it had already trained on, and the validation error would no longer measure how it does on engines it has never seen. Usually it would come out smaller, but nothing guarantees the direction: a split by window would also hold back different windows and train a different network on the rest, and this project has never built one to measure the effect. (Corrected 2026-10-02: this said the validation error would be far too optimistic, which is the usual direction but not a certain one, and that neighbouring labels always differ by one, which is not so at the ceiling. The first was found by the search for that kind of claim after the review of the playground's information cards on 2026-09-26; the second on 2026-10-02 while this cell was being corrected, by printing the labels the default preparation gives engine 1's first two windows, 125 and 125.)

Splitting whole engines keeps the two sets genuinely separate.

**Holding engines back is this project's addition** (assumption A-06). The paper describes no validation split at all; its Table 6 states a fixed 200 epochs with no early stopping. A number produced with held-back engines therefore follows a different procedure from the paper's, and has to be described that way wherever it is set beside a published one.

In [ ]:
# The engines were split in step 6, before any statistic was fitted.
# Here that decision is applied to the windows.
train_windows = train_windows_all.subset_by_units(train_units)
validation_windows = train_windows_all.subset_by_units(validation_units)

print()
print(f"Training   : {len(train_windows):>6} windows from {len(train_units)} engines")
print(f"Validation : {len(validation_windows):>6} windows from {len(validation_units)} engines")
print(f"Test       : {len(test_windows):>6} windows from {len(np.unique(test_windows.units))} engines")

overlap = set(train_windows.units.tolist()) & set(validation_windows.units.tolist())
assert not overlap, f"Engines in both splits: {sorted(overlap)}"
print("\nOK  No engine appears in both splits.")
print(f"OK  All windows accounted for: "
      f"{len(train_windows)} + {len(validation_windows)} = {len(train_windows_all)}")

In [ ]:
# Do the two splits carry the same kind of engine? Drawn as two outlines rather
# than two see-through fills, because overlapping fills make a third colour that
# belongs to neither. The ceiling gets its own panel: it holds a third of the
# windows, so on a shared axis it flattens everything else.
fig, (spread_axis, ceiling_axis) = plt.subplots(
    1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [2.4, 1]})

bins = np.arange(0, config.rul_max + 5, 5)
splits = (
    ("training", train_windows.targets, "#4C72B0"),
    ("validation", validation_windows.targets, "#DD8452"),
)

for name, targets, colour in splits:
    below = targets[targets < config.rul_max]
    # Share of ALL that split's windows, so the two panels together add to 100 %.
    weights = np.full(len(below), 1 / len(targets))
    spread_axis.hist(below, bins=bins, weights=weights, histtype="step",
                     linewidth=2, color=colour,
                     label=f"{name} ({len(targets):,} windows)")

spread_axis.set_xlabel("Remaining Useful Life (cycles)")
spread_axis.set_ylabel("share of that split's windows")
spread_axis.set_title("Labels below the ceiling: flat in both splits, and "
                      "the same shape", fontsize=10.5)
spread_axis.annotate("validation runs a little higher here because less of\n"
                     "it sits at the ceiling, so more of it is spread below",
                     xy=(0.03, 0.12), xycoords="axes fraction", fontsize=8.5,
                     color="#555555")
spread_axis.legend(fontsize=8)
spread_axis.set_ylim(0, None)

shares = [float((targets >= config.rul_max).mean()) for _, targets, _ in splits]
bars = ceiling_axis.bar([name for name, _, _ in splits], shares,
                        color=[colour for _, _, colour in splits], width=0.55)
for bar, share in zip(bars, shares):
    ceiling_axis.text(bar.get_x() + bar.get_width() / 2, share + 0.012,
                      f"{share:.1%}", ha="center", fontsize=9.5)
ceiling_axis.set_ylim(0, max(shares) * 1.35)
ceiling_axis.set_ylabel("share of that split's windows")
ceiling_axis.set_title(f"Windows sitting at the ceiling ({config.rul_max})",
                       fontsize=10.5)

fig.tight_layout()
plt.show()

gap = abs(train_windows.targets.mean() - validation_windows.targets.mean())
print(f"Mean label, training   : {train_windows.targets.mean():6.2f} cycles")
print(f"Mean label, validation : {validation_windows.targets.mean():6.2f} cycles")
print(f"Difference             : {gap:6.2f} cycles")
print()
print("Below the ceiling both lines are flat: inside that range every label is")
print("about as common as every other, in both splits, which is what a split")
print("by engine should give. The validation line sits roughly a tenth higher")
print("throughout, and that is not a second finding -- it is the same one seen")
print("from the other side. Each split's bars add up to 100 %, so a split with")
print("less at the ceiling has more to spread below it.")
print()
print(f"They part at the ceiling: {shares[0]:.1%} of training windows sit there")
print(f"against {shares[1]:.1%} of validation windows. A window reaches the")
print("ceiling only if its engine was still healthy, so the held-back 20 engines")
print("are the shorter-lived ones, and that alone explains the "
      f"{gap:.2f}-cycle gap in the means.")
print()
print("That is worth knowing rather than hiding: it is a fact about this seed's")
print("split, and it is why a validation score and a test score are not")
print("comparable numbers (A-06).")

## 5. The sensor graph

This is the part of the paper that is **fully specified**. Its Table 3 prints the entire 14-by-14 adjacency matrix, so nothing has to be assumed here — only transcribed correctly, which is checked automatically.

### How the authors built it

A graph for this dataset can also be built from the data, for example by joining the sensors whose readings move together most strongly: `notebooks/graph/03_correlation_fd001.ipynb` builds one such graph, and `notebooks/graph/04_comparison_fd001.ipynb` sets the networks trained on it beside those trained on the paper's. Wen et al. designed theirs by hand, from knowledge of how a turbofan works. Their five rules:

1. **Same location.** Sensors at the same engine station are connected, because they measure the same gas at the same point. Example: `P30` and `Ps30`, the total and static pressure at the high-pressure compressor outlet.
2. **Speed correction.** A raw speed is connected to its corrected counterpart, because one is computed from the other. Example: `Nf` and `NRf`.
3. **Gas path.** Sensors are connected along the route the gas actually takes. Example: the temperature chain `T24` → `T30` → `T50`, following the flow from low-pressure compressor to high-pressure compressor to low-pressure turbine.
4. **Fuel and control.** Fuel quantities are connected to what they influence. Example: `phi`, the ratio of fuel flow to static pressure, is connected to the temperatures and pressures that burning more fuel changes.
5. **Cooling.** The coolant bleeds are connected to each other, because they jointly manage turbine temperature.

### Why it must be normalised before use

A graph convolution layer does not use the raw matrix $A$. Two forms fit what the paper says, and they begin the same way:

$$\hat{A}_{\text{sym}} = D^{-1/2}\,(A + I)\,D^{-1/2}\qquad\qquad \hat{A}_{\text{row}} = D^{-1}\,(A + I)$$

Two things happen in either one:

- Adding the identity matrix $I$ joins every node to itself, so that a node keeps its own current value instead of being rebuilt out of its neighbours alone.
- Dividing by the **degrees** (the number of edges at each node, held on the diagonal of $D$) stops well-connected nodes from dominating. Without this, repeatedly summing over neighbours makes values grow layer after layer and training becomes unstable.

**Which form to use is this project's choice** (assumption A-21). The paper describes this step but never prints the formula. This project uses the **row** form: `config.adjacency_normalization = "row"`. The difference between the two is what each node ends up receiving. Under the row form every node's incoming weights add up to exactly 1, so each node takes in one unit of information divided among itself and its neighbours; the symmetric form does not add up to 1, and stays symmetric instead.

The cell below draws the form the pipeline saves and the network is given, rather than the more commonly quoted symmetric one, so that what is shown here is what the model actually receives.

In [ ]:
adjacency, symbols = build_physics_adjacency()
report = adjacency_report(adjacency, symbols)

print("Properties of the sensor graph")
print("-" * 50)
for key in ["n_nodes", "n_edges", "density", "is_symmetric", "has_self_loops", "is_connected"]:
    value = report[key]
    print(f"  {key:16s} {value:.4f}" if isinstance(value, float) else f"  {key:16s} {value}")

print("\nDegree of each sensor (how many neighbours it has):")
degrees = pd.Series(report["degrees"]).sort_values(ascending=False)
print(degrees.to_string())

assert report["is_symmetric"], "Transcription error: the matrix is not symmetric."
assert report["is_connected"], "The graph is not connected."
assert report["n_edges"] == 41, "Edge count differs from the published table."
print("\nOK  Symmetric, connected, 41 undirected edges - matches the paper's Table 3.")

In [ ]:
normalized_adjacency = normalize_adjacency(
    adjacency, method=config.adjacency_normalization
)

# Written out rather than hard-coded, so the label stays true if the
# setting changes.
FORMULA = {
    "row": r"$\hat{A} = D^{-1}(A+I)$",
    "symmetric": r"$\hat{A} = D^{-1/2}(A+I)D^{-1/2}$",
}[config.adjacency_normalization]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))

for axis, matrix, title in [
    (axes[0], adjacency, "Raw adjacency matrix (Table 3)\n1 = connected, 0 = not"),
    (axes[1], normalized_adjacency,
     f"After normalisation, {config.adjacency_normalization} form (A-21)\n"
     f"{FORMULA}, which is what the network is given"),
]:
    image = axis.imshow(matrix, cmap="Blues", vmin=0)
    axis.set_xticks(range(14)); axis.set_xticklabels(symbols, rotation=90, fontsize=7)
    axis.set_yticks(range(14)); axis.set_yticklabels(symbols, fontsize=7)
    axis.set_title(title, fontsize=9)
    fig.colorbar(image, ax=axis, fraction=0.046)

fig.tight_layout()
plt.show()

print("Notice the diagonal in the right-hand panel: on the left it is")
print("empty, because Table 3 never joins a sensor to itself, and on the")
print("right it is filled, because the formula adds that join.")
print()
print("Row sums after normalisation:",
      normalized_adjacency.sum(axis=1)[:4].round(3))
print("Under the row form each one is exactly 1: a node takes in a single")
print("unit of information, shared out between itself and its neighbours.")


In [ ]:
# The graph drawn as a network, with node size showing degree.
angles = np.linspace(0, 2 * np.pi, 14, endpoint=False)
positions = np.column_stack([np.cos(angles), np.sin(angles)])

fig, axis = plt.subplots(figsize=(6.2, 6.2))
for i in range(14):
    for j in range(i + 1, 14):
        if adjacency[i, j]:
            axis.plot(*zip(positions[i], positions[j]), color="#B8C4D9", linewidth=0.9, zorder=1)

degree_values = adjacency.sum(axis=1)
scatter = axis.scatter(positions[:, 0], positions[:, 1], s=degree_values * 90,
                       c=degree_values, cmap="YlOrRd", edgecolors="#333333",
                       linewidths=0.8, zorder=2)
for position, symbol in zip(positions, symbols):
    axis.annotate(symbol, position * 1.17, ha="center", va="center", fontsize=8.5)

axis.set_xlim(-1.42, 1.42); axis.set_ylim(-1.42, 1.42)
axis.set_aspect("equal"); axis.axis("off")
axis.set_title("The physics-based sensor graph\n(node size and colour = number of neighbours)",
               fontsize=10)
fig.colorbar(scatter, ax=axis, label="degree", fraction=0.04)
fig.tight_layout()
plt.show()

## 6. The whole thing in one call

Every step above is also available as a single function. This is what a script or an automated test uses, and it is what guarantees the notebook and the rest of the project do exactly the same thing.

The next cell runs it and checks that the result is identical to what was built step by step. If those two ever disagree, the notebook and the package have drifted apart and one of them is wrong.

In [ ]:
prepared = run_preprocessing(config, data_dir=data_dir, keep_frames=False)
print()
display(prepared.summary())

In [ ]:
np.testing.assert_array_equal(prepared.train.features, train_windows.features)
np.testing.assert_array_equal(prepared.train.targets, train_windows.targets)
np.testing.assert_array_equal(prepared.test.features, test_windows.features)
np.testing.assert_array_equal(prepared.validation.units, validation_windows.units)
print("OK  The one-call pipeline reproduces the step-by-step result exactly.")

## 7. Final verification

Every claim this notebook makes, checked in one place. A reader who trusts nothing else should be able to run this cell.

In [ ]:
checks: list[tuple[str, bool, str]] = []

def check(description: str, condition: bool, detail: str = "") -> None:
    checks.append((description, bool(condition), detail))

# --- shapes and alignment ---
n_nodes = prepared.adjacency_raw.shape[0]
check("Feature axis matches the graph size",
      prepared.train.features.shape[1] == n_nodes, f"{prepared.train.features.shape[1]} nodes")
check("Sensor order matches the paper's Table 3",
      list(prepared.sensor_symbols) == list(PAPER_SENSOR_SYMBOLS))
# The pipeline's job is to hand over the window intact. Turning it into the
# one number Table 4 prints is the model's job, done by the temporal stage
# (A-07), so that is checked in notebook 02 and not here.
check("Each node carries its whole window (A-07)",
      prepared.train.features.shape[2] == config.window_size,
      f"{prepared.train.features.shape[2]} cycles per sensor")

# --- labels ---
for name, dataset in [("training", prepared.train), ("validation", prepared.validation),
                      ("test", prepared.test)]:
    check(f"Labels within [0, {config.rul_max}] ({name})",
          dataset.targets.min() >= 0 and dataset.targets.max() <= config.rul_max,
          f"[{dataset.targets.min():.0f}, {dataset.targets.max():.0f}]")

# --- data hygiene ---
check("No missing or infinite feature values",
      all(np.isfinite(d.features).all()
          for d in (prepared.train, prepared.validation, prepared.test)))
check("Training and validation engines are disjoint",
      set(prepared.train.units.tolist()).isdisjoint(prepared.validation.units.tolist()))
check("Exactly one window per test engine",
      len(prepared.test) == len(np.unique(prepared.test.units)), f"{len(prepared.test)} engines")

# --- the graph ---
graph = prepared.diagnostics["graph"]
check("Graph is symmetric", graph["is_symmetric"])
check("Graph is connected", graph["is_connected"])
check("Graph has 41 edges, as published", graph["n_edges"] == 41, f"{graph['n_edges']} edges")
check("No isolated sensors", not graph["isolated_nodes"])
check("Normalised graph has self-loops",
      bool(np.all(np.diag(prepared.adjacency_normalized) > 0)))

# --- report ---
width = max(len(description) for description, _, _ in checks) + 2
print(f"{'CHECK':<{width}} RESULT   DETAIL")
print("-" * (width + 26))
for description, passed, detail in checks:
    print(f"{description:<{width}} {'PASS' if passed else 'FAIL':<8} {detail}")

failed = [description for description, passed, _ in checks if not passed]
print("-" * (width + 26))
if failed:
    raise AssertionError(f"{len(failed)} check(s) failed: {failed}")
print(f"All {len(checks)} checks passed.")

## 8. Save the result

Five files are written, so the modelling stage never has to repeat this work and so the exact inputs of any experiment can be recovered later:

| File | Contents |
|---|---|
| `windows.npz` | The three splits as arrays |
| `adjacency.npz` | The sensor graph, raw and normalised |
| `normalization_statistics.csv` | The learned scaling values |
| `config.yaml` | The exact settings used |
| `diagnostics.json` | What the pipeline observed |

Saving the settings alongside the arrays is what makes a result traceable. Six months from now, `config.yaml` is what answers the question "which version of the preprocessing produced this?"

In [ ]:
output_dir = PROJECT_ROOT / "data" / "processed"
saved_to = prepared.save(output_dir)

print(f"\nSaved to: {saved_to}\n")
for path in sorted(saved_to.iterdir()):
    print(f"  {path.name:32s} {path.stat().st_size / 1024:8.1f} KB")

In [ ]:
# Confirm the saved arrays can be read back and are identical.
with np.load(saved_to / "windows.npz", allow_pickle=False) as archive:
    print("Arrays in windows.npz:")
    for key in sorted(archive.files):
        print(f"  {key:20s} {archive[key].shape!s:>18}  {archive[key].dtype}")
    np.testing.assert_array_equal(archive["train_features"], prepared.train.features)

print("\nOK  Saved arrays match what is in memory.")

## 9. Summary and next steps

### What this notebook produced

Starting from three plain text files, it produced:

- **Training, validation and test examples** of shape `(N, 14, 30)`: 14 sensor nodes, each carrying its whole scaled, smoothed window. How many of those 30 numbers a node takes into the first graph layer is decided by the model, in notebook 02.
- **A 14-node sensor graph**, transcribed from the paper's Table 3 and verified symmetric, connected, and with 41 edges.
- **A record of every decision** the paper does not specify.

### Decisions that were not read from the paper

| Decision | Value used | Written up as |
|---|---|---|
| Label ceiling | 125 cycles | A-01 |
| Ceiling applied to test labels | Yes | A-02 |
| Input rescaling | z-score — subtract the mean, divide by the spread | A-03 |
| Smoothing strength | 0.5 | A-04 |
| Window stride | 1 cycle | A-05 |
| Held-back engines | 20 % of the training engines, chosen by engine | A-06 |
| Smoothing formula variant | The recursive form of the paper's Equation 18 | A-09 |
| Seventh dropped sensor | `NRc`, deduced from Table 3 | A-10 |
| Graph rescaling | Each row divided by its own total | A-21 |
| Where the scaling is fitted | After the split, on the training engines only | A-25 |

Full justifications are in `docs/assumptions/`. **One more decision shapes every result that follows**, although it belongs to the model rather than to this notebook: the network notebook 02 trains passes each sensor's most recent reading to the graph layers instead of the paper's recurrent stage. So its results compare with the paper's own **"ESTGNN w/o LSTM"** row, not with its headline figures.

### What comes next

**Notebook 02** carries on from here, and so does the numbering. It rebuilds these same arrays and this same graph in one call, then:

- **step 9** builds the network of the paper's Table 4,
- **step 10** trains it on the 80 training engines, using the 20 held back to decide when to stop,
- **step 11** scores it on the 100 test engines with the paper's four measures.

What happens before the graph layers has a folder of its own too, `notebooks/temporal/`. The network's first part, its **temporal stage**, turns each sensor's 30 readings into the number or numbers that sensor's graph node carries; notebook 02's network keeps only the newest reading. That folder has one notebook for each of the six stages this project builds, the paper's recurrent one among them, and a seventh, `07_comparison_fd001.ipynb`, that sets their stored five-seed results side by side.

What the sensor graph does has a folder of its own, `notebooks/graph/`. It shows the paper's graph and two others on the real data — every sensor joined to every other, and the 41 pairs of sensors whose readings move together most strongly — and its last notebook, `04_comparison_fd001.ipynb`, compares the networks trained on the three. Explaining the trained network's predictions is the job of `notebooks/xai/`.

(Corrected 2026-10-03: this paragraph and section 5 said that notebook 02 sets the paper's graph beside others. That comparison moved from notebook 02 to `notebooks/graph/` on 2026-10-02, and the two pointers here were found stale when that folder was built.)

### References

- Corso, G., et al. (2024). Graph neural networks. *Nature Reviews Methods Primers*, 4, 17.
- Heimes, F. O. (2008). Recurrent neural networks for remaining useful life estimation. *PHM 2008*.
- Jin, Y., et al. (2025). Heterogeneous dynamic-aware GNN for RUL prediction of aeroengine. *IEEE TIM*, 74, Article 3538014.
- Kapoor, S., & Narayanan, A. (2023). Leakage and the reproducibility crisis. *Patterns*, 4(9), 100804.
- Kipf, T. N., & Welling, M. (2017). Semi-supervised classification with graph convolutional networks. *ICLR*.
- Li, X., Ding, Q., & Sun, J.-Q. (2018). Remaining useful life estimation using deep convolution neural networks. *RESS*, 172, 1-11.
- Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions. *NeurIPS 30*.
- MacQueen, J. (1967). Some methods for classification and analysis of multivariate observations. In L. M. Le Cam & J. Neyman (Eds.), *Proceedings of the Fifth Berkeley Symposium on Mathematical Statistics and Probability: Vol. 1. Statistics* (pp. 281–297). University of California Press.
- Saxena, A., et al. (2008). Damage propagation modeling for aircraft engine run-to-failure simulation. *PHM 2008*.
- Shapley, L. S. (1953). A value for n-person games. *Contributions to the Theory of Games II*.
- Wen, J.-R., et al. (2026). Explainable spatiotemporal graph neutral network with dynamic SHAP enhancement. *Aerospace Science and Technology*, 168, 110875.

Full citations, with a note on what each is used for, are in `docs/REFERENCES.md`.

Until 2026-10-02 the MacQueen entry above lacked its editors and pages, and the Jin et al. entry its article number. Both were found when the cited papers were collected into `docs/references/` and checked against the places that cite them.